# 대학교 약어(연대, 건대 등) 사전 — 규칙 시도 후 하드코딩으로 전환

**시도했던 것**: `~대학교`에서 "학교"를 떼고 첫 글자 + "대"를 약어로 자동 생성해보려 했음
(예: 연세대학교 -> 연대). 근데 한국 대학교 이름이 "서울~", "한국~", "동~" 같은 흔한 접두어에
몰려있어서, 첫 글자 하나만으로는 전혀 구분이 안 됨. 실제로 확인해보니 "동"으로 시작하는
대학교가 18개, "서"로 시작하는 게 27개, "한"으로 시작하는 게 41개라 전부 같은 약어로
충돌해버림 — 정작 필요한 건대/고대/동대/서울대/연대/한양대 같은 유명 대학들이 전부
이 충돌 때문에 자동 생성에서 빠지고, 아무도 안 쓰는 희귀한 대학 이름들만 살아남는
역설적인 결과가 나옴.

**결론**: 규칙으로 일반화하는 걸 포기하고, 직접 아는 약어를 하드코딩하기로 함.

In [20]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src" / "data") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src" / "data"))

REPO_ROOT

WindowsPath('d:/Study/dongguk_university/dreampath')

In [21]:
import pandas as pd

gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
gt_df.shape

(12439, 2)

## (참고 기록) 시도했던 규칙 + 충돌 증거

최종 결과물엔 안 쓰지만, 왜 규칙을 포기했는지 근거로 남겨둠.

In [ ]:
def generate_abbreviation(name: str) -> str | None:
    if not name.endswith("대학교"):
        return None
    stem = name[:-2]
    if len(stem) < 2:
        return None
    return stem[0] + "대"


abbrev_df = gt_df.assign(약어=gt_df["학교명"].apply(generate_abbreviation)).dropna(subset=["약어"])
counts = abbrev_df.groupby("약어")["학교명"].nunique().sort_values(ascending=False)
print("충돌(같은 약어에 여러 대학) 상위 사례:")
counts[counts > 1].head(10)

## 하드코딩 별칭 사전

직접 확인한 통용 약어만 수동으로 등록. 새로운 게 발견되면 이 딕셔너리에 계속 추가.

In [22]:
MANUAL_ALIASES = {
    "연세대학교": ["연대"],
    "고려대학교": ["고대"],
    "이화여자대학교": ["이대", "이화여대"],
    "한국외국어대학교": ["외대", "한국외대"],
    "성균관대학교": ["성대"],
    "홍익대학교": ["홍대"],
    "숙명여자대학교": ["숙대"],
    "동덕여자대학교": ["동덕여대"],
    "성신여자대학교": ["성신여대"],
    "중앙대학교": ["중대"],
    "동국대학교": ["동대"],
    "건국대학교": ["건대"],
    # 아래 부속학교 항목들은 gt_match.ipynb를 한 번 돌려서 gt_match_count==0인 행들을 확인하다가
    # 발견함 (정식명이 "인하대학교사범대학부속중학교"처럼 불규칙해서 접미사 규칙으로 못 뽑힘).
    # 실제 comment에서 확인된 축약 변형을 전부 나열. 인하대는 부속 초등학교가 GT에 아예 없어서 대상 없음.
    "이화여자대학교사범대학부속초등학교": ["이화여대부속초", "이대부초"],
    "이화여자대학교사범대학부속이화·금란중학교": ["이화여대부속중", "이대부중"],
    "이화여자대학교사범대학부속이화금란고등학교": ["이화여대부속고", "이대부고"],
    "인하대학교사범대학부속중학교": ["인하대부속중", "인하부중", "인하대학교부속중"],
    "인하대학교사범대학부속고등학교": ["인하대부속고", "인하부고", "인하대학교부속고"],
}

# 학교당 별칭 개수가 제각각이라 약어/약어2로 나누지 않고, 한 컬럼에 공백으로 이어붙임
gt_df["약어"] = gt_df["학교명"].map(lambda n: " ".join(MANUAL_ALIASES.get(n, [])) or None)
gt_df = gt_df.drop(columns=["약어2"], errors="ignore")

out_path = REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv"
gt_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"약어 채워진 행: {gt_df['약어'].notna().sum()}개")
out_path

약어 채워진 행: 17개


WindowsPath('d:/Study/dongguk_university/dreampath/data/processed/gt_schoolnames.csv')